In [ ]:

# 1. Environment & Path Setup
import os
import sys

# Add the project root directory to sys.path so 'pipeline' is importable
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
from statsbombpy import sb

from pipeline import get_events_from_timeline, get_complete_carries, spatial_unpack

In [ ]:
match_df = get_complete_carries(spatial_unpack(get_events_from_timeline(1, 3754296))).sort_values(by = 'match_seconds').reset_index(drop = True)

In [ ]:
match_df = spatial_unpack(match_df, targetcolumn = 'pass_end_location', x_name = 'pass_x', y_name = 'pass_y')

In [ ]:
def get_carry_outcome(event_df):
    RELEASE_EVENTS = ['Pass', 'Shot', 'Complete Carry']

    df = event_df.copy()
    main_release = df[df['type'].isin(RELEASE_EVENTS)].copy()

    next_release = main_release.shift(-1).copy()
    time_diff = next_release['match_seconds'] - main_release['carry_end_seconds']
    spatial_diff = np.sqrt(
        ((main_release['end_x'] - next_release['x'])**2) +
        ((main_release['end_y'] - next_release['y'])**2)
    )
    player_diff = (main_release['player_id'] == next_release['player_id'])
    next_type = next_release['type']

    chain_continues = (time_diff <= 0.150) & (spatial_diff <= 0.5) & (player_diff)
    chain_starts = ~chain_continues.shift(1, fill_value=False)
    main_release['release_chain_id'] = chain_starts.cumsum()
    main_release['next_type'] = next_type

    next_release_2 = main_release.shift(-1).copy()
    chain_match = (main_release['release_chain_id'] == next_release_2['release_chain_id'])
    main_release['release_next_chain'] = chain_match

    main_release = main_release[main_release['type'] == 'Complete Carry']

    main_release['carry_end_outcome'] = np.where(
        main_release['release_next_chain'] == True, 
        main_release['next_type'], 
        'Other'
    )

    df['carry_outcome'] = np.nan
    carry_mask = df['type'].to_numpy() == 'Complete Carry'

    df.loc[carry_mask, 'carry_end_outcome'] = main_release['carry_end_outcome'].to_numpy()

    return df


In [ ]:
match_updated_df = get_carry_outcome(match_df)

In [ ]:
print(match_df['type'].unique())

In [ ]:
ff_events = match_df['50_50']

In [ ]:
ff_normalised = pd.json_normalize(ff_events)

In [ ]:
match_df[['50_50_outcome_id', '50_50_outcome']] = ff_normalised

In [ ]:
ff_true = match_df[match_df['type'] == '50/50']

In [ ]:
key_events = ['Ball Recovery', 'Ball Receipt*', 'Interception', 'Complete Carry', 'Offside', '50/50', 'Pass', 'Shot', 'Duel']

In [ ]:
key_match_df = match_df[match_df['type'].isin(key_events)]

In [ ]:
print(key_match_df['type'].unique())

In [ ]:
key_columns = ['player', 'match_seconds', 'play_pattern', 'type','location', 'interception_outcome', 'duel_outcome', '50_50', 'carry_end_location', 'pass_end_location', 'x', 'y','end_x', 'end_y', 'pass_x', 'pass_y']

In [ ]:
main_df = key_match_df[key_columns]

In [ ]:
main_df = main_df.sort_values(by = 'match_seconds').reset_index(drop = True)

In [ ]:
main_ff_check = main_df[main_df['type'] == '50/50']

Ball Receptions:
- Ball Receipt*
- Ball Recovery
- Duel(Won)
- 50/50(Won)
- Interception(Won)
- Pass(Recovery)
- Pass(Recovery)

Won/Recovery columns:
- duel_outcome == 'Won'
- interception_outcome == 'Won'
- 50_50_outcome == 'Won'
- pass_type == 'Recovery'
- shot_type == 'Recovery'

In [ ]:
reception_types = ['Ball Receipt*', 'Ball Recovery']
open_play_phases = ['Regular Play', 'From Throw In', 'From Counter', 'From Keeper']
main_receptions = match_df[(match_df['type'].isin(reception_types)) |
                           ((match_df['type'] == 'Duel') & (match_df['duel_outcome'] == 'Won')) |
                           ((match_df['type'] == 'Interception') & (match_df['interception_outcome'] == 'Won')) |
                           ((match_df['type'] == '50_50') & (match_df['50_50_outcome'] == 'Won')) |
                           ((match_df['type'] == 'Pass') & (match_df['pass_type'] == 'Recovery')) |
                           ((match_df['type'] == 'Shot') & (match_df['shot_type'] == 'Recovery'))
]
main_receptions = main_receptions[main_receptions['play_pattern'].isin(open_play_phases)]

In [ ]:
def average_y(event_df, step_length = 1, window_rad = 2.5):
    df = event_df.copy()
    df = df.sort_values(by = ['x']).reset_index(drop=True)
    x_grid = np.arange(0, 121, step=step_length)

    x_arr = df['x'].values
    y_arr = df['y'].values

    y_cumsum = np.cumsum(np.insert(y_arr, 0, 0))
    left_indices = np.searchsorted(x_arr, x_grid - window_rad, side="left")
    right_indices = np.searchsorted(x_arr, x_grid + window_rad, side="right")

    counts = right_indices - left_indices
    sums = y_cumsum[right_indices] - y_cumsum[left_indices]

    with np.errstate(divide="ignore", invalid="ignore"):
        avg_y = np.where(counts > 0, sums / counts, np.nan)

    return np.column_stack((x_grid, avg_y))

In [ ]:
av_y_receptions = average_y(main_receptions)

In [ ]:
def y_quad_average(event_df, step_length = 1, window_rad = 2.5):
    df = event_df.copy()
    xy_grid = average_y(df, step_length=step_length, window_rad=window_rad)

    x_vals = xy_grid[:, 0]
    y_vals = xy_grid[:, 1]
    
    fit_mask = ~np.isnan(y_vals)
    fit_x = x_vals[fit_mask]
    fit_y = y_vals[fit_mask]

    if len(fit_x) < 3:
        raise ValueError("Not enough valid window averages to fit a quadratic curve.")

    a, b, c = np.polyfit(fit_x, fit_y, deg=2)
    return [a, b, c]

In [ ]:
quad_y_receptions = y_quad_average(main_receptions)

In [ ]:
def get_normalised_y(event_df, step_length = 1, window_rad = 2.5):
    UPPER = 120
    LOWER = 0

    df = event_df.copy()
    coeffs = y_quad_average(df, step_length=step_length, window_rad=window_rad)

    df['curve_y'] = (coeffs[0] * (df['x']**2)) + (coeffs[1] * df['x']) + (coeffs[2])
    df["y_distance"] = df["y"] - df["curve_y"]

    above_curve = df["y"] >= df["curve_y"]
    df["boundary_distance"] = np.where(
        above_curve,
        np.abs(UPPER - df["curve_y"]),
        np.abs(df["curve_y"] - LOWER),
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        df["normalised_y"] = np.where(
            df["boundary_distance"] > 0,
            np.abs(df["y_distance"] / df["boundary_distance"]),
            0,
        )

    return df

In [ ]:
normalised_receptions = get_normalised_y(main_receptions)

In [ ]:
release_events = ['Pass', 'Shot', 'Complete Carry']
main_release = match_df[match_df['type'].isin(release_events)].sort_values(by = 'match_seconds').dropna(axis = 1, how = 'all').reset_index(drop = True)

In [ ]:
view_columns = ['player', 'type', 'match_seconds', 'location', 'carry_end_location', 'carry_end_matchseconds']
main_release_view = main_release[view_columns]

In [ ]:
next_release = main_release.shift(-1).copy()
time_diff = next_release['match_seconds'] - main_release['carry_end_seconds']
spatial_diff = np.sqrt(
    ((main_release['end_x'] - next_release['x'])**2) +
    ((main_release['end_y'] - next_release['y'])**2)
)
player_diff = (main_release['player_id'] == next_release['player_id'])
next_type = next_release['type']

In [ ]:
chain_continues = (time_diff <= 0.150) & (spatial_diff <= 0.5) & (player_diff)
chain_starts = ~chain_continues.shift(1, fill_value=False)
main_release['chain_id'] = chain_starts.cumsum()
main_release['next_type'] = next_type

In [ ]:
next_release_2 = main_release.shift(-1).copy()
chain_match = (main_release['chain_id'] == next_release_2['chain_id'])
main_release['next_chain'] = chain_match

In [ ]:
main_release_view_2 = main_release[['type', 'next_type', 'next_chain']]
main_release_view_3 = main_release_view_2[main_release_view_2['type'] == 'Complete Carry']
main_release_view_4 = main_release_view_2[main_release_view_2['type'].isin(['Pass', 'Shot'])]